# Lekcia 11 - Protokol modelového kontextu (MCP)

**Protokol modelového kontextu (MCP)** je otvorený štandard, ktorý umožňuje agentom dynamicky objavovať a používať nástroje, zdroje a dátové zdroje počas behu programu. Namiesto pevného zabudovania nástrojov do agenta, MCP umožňuje agentom pripojiť sa k externým serverom, ktoré na požiadanie sprístupňujú svoje schopnosti.

V tejto lekcii sa naučíte:
- Čo je MCP a prečo je dôležitý pre agentské systémy
- Ako funguje architektúra klient-server MCP
- Ako vytvoriť agentov, ktorí používajú objavovanie nástrojov podľa MCP


## Nastavenie

**Predpoklady:**
- Projekt Microsoft Foundry s nasadeným modelom
- Spustite `az login` pre autentifikáciu


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Čo je Protokol Kontextu Modelu (MCP)?

MCP definuje štandardný spôsob, ako môžu AI agenti objavovať a interagovať s externými nástrojmi a zdrojmi dát:

- **MCP Server**: Zverejňuje nástroje, zdroje a výzvy cez štandardný protokol
- **MCP Klient**: Runtime agenta, ktorý sa pripája na servery a objavuje dostupné schopnosti
- **Dynamické objavovanie**: Agenti nepotrebujú pevne zakódované nástroje — objavujú, čo je dostupné počas behu

Toto je silný nástroj na budovanie rozšíriteľných systémov agentov, kde je možné pridávať nové schopnosti bez úpravy kódu agenta.


## Ako MCP funguje

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (klient MCP) sa pripojí na MCP server
2. Server odpovie zoznamom dostupných nástrojov a ich schém
3. Agent potom môže počas svojho uvažovania vyvolať akýkoľvek zistený nástroj
4. Výsledky sa vracajú späť rovnakým protokolom


## Simulácia zisťovania nástrojov MCP

Pretože skutočný server MCP vyžaduje bežiaci serverový proces, ukážeme si vzor pomocou funkcií `@tool`, ktoré simulujú to, čo by poskytovala služba ubytovania pripojená k MCP.

V produkcii by boli tieto nástroje dynamicky zisťované zo servera MCP namiesto toho, aby boli definované lokálne.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Vytváranie agenta s nástrojmi štýlu MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP v produkcii

V produkčnom prostredí MCP umožňuje výkonné vzory:

- **Dynamické objavovanie nástrojov**: Agenti sa pripájajú k MCP serverom a za behu objavujú nástroje
- **Oddelená architektúra**: Poskytovatelia nástrojov môžu aktualizovať nezávisle od agenta
- **Zdieľanie naprieč organizáciami**: Tímy môžu sprístupňovať schopnosti cez MCP servery, ktoré môže používať akýkoľvek agent
- **Podpora Microsoft Agent Framework**: MAF má zabudovanú podporu MCP klienta cez integráciu `mcp`

Ak chcete použiť skutočný MCP server s MAF, pripojíte sa cez `hosted_mcp_tool()` alebo integráciu MCP klienta.

**Viac informácií:**
- [Špecifikácia MCP](https://modelcontextprotocol.io/)
- [Podpora Microsoft Agent Framework MCP](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Zhrnutie

V tejto lekcii ste sa naučili:
- **MCP** je otvorený štandard pre dynamické zisťovanie nástrojov medzi agentmi a poskytovateľmi nástrojov
- **Klient-server architektúra** umožňuje agentom zisťovať schopnosti za behu
- MCP umožňuje **rozšíriteľné, oddelené systémy agentov**, kde je možné pridávať nástroje bez zmien kódu
- Microsoft Agent Framework poskytuje **vstavanú podporu MCP** pre produkčné použitie


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
